# YOLO Bake-off Trainer (Google Colab)

Train the **player** and **class-agnostic logo** YOLOv8 detectors used in Workflow 1, then download the weights back to your Mac.

## How to use

1. Upload this file to [Google Colab](https://colab.research.google.com/) (**File → Upload notebook**), or open it from Drive.
2. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is fine).
3. Run cells top to bottom (**Runtime → Run all**).
4. At the end, Colab downloads `yolo_bakeoff_weights.zip` (or use the Drive copy).
5. On your Mac, unzip into:

```text
local-data/groundingdino/data_collection/weights/yolo/
  player_detector/best.pt
  logo_detector/best.pt
```

**Expected time (Colab T4):** logo ~5–15 min (5 epochs), player ~30–90 min (25 epochs).

## 0. GPU check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU — go to Runtime → Change runtime type → GPU, then Runtime → Restart session.")

In [ ]:
%pip -q install ultralytics roboflow pyyaml opencv-python-headless

## 1. Config + folders

Optional: set `MOUNT_DRIVE = True` to also save weights under Google Drive.

In [ ]:
from pathlib import Path
import json
import shutil
import yaml
import torch
from ultralytics import YOLO
from roboflow import Roboflow

MOUNT_DRIVE = False  # True → also copy zip to Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path("/content/yolo_bakeoff")
ROBOFLOW_DIR = ROOT / "roboflow"
MERGED_LOGO_DIR = ROOT / "merged_logo_detector"
RUNS_DIR = ROOT / "runs"
WEIGHTS_OUT = ROOT / "weights" / "yolo"
PLAYER_WEIGHTS = WEIGHTS_OUT / "player_detector" / "best.pt"
LOGO_WEIGHTS = WEIGHTS_OUT / "logo_detector" / "best.pt"

for p in (ROOT, ROBOFLOW_DIR, RUNS_DIR, WEIGHTS_OUT):
    p.mkdir(parents=True, exist_ok=True)

ROBOFLOW_API_KEY = "A1UDb8W87l9RzmQ4CMlU"
PLAYER_WORKSPACE = "school-hnpey"
PLAYER_PROJECT = "soccer-iqjx5"
PLAYER_VERSION = 2
LOGO_DETECTION_SOURCES = [
    ("sponsor_detector", "syed-fathaullah", "sponsor-detector", 2),
    ("logo_detection_e92p3", "data-deo7j", "logo-detection-e92p3", 4),
]

PLAYER_EPOCHS = 25
LOGO_EPOCHS = 5
IMGSZ = 640
DEVICE = 0 if torch.cuda.is_available() else "cpu"
PLAYER_NAME_HINTS = {"joueur", "gardien", "player", "goalkeeper", "keeper"}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

print("ROOT =", ROOT)
print("DEVICE =", DEVICE)

## 2. Download Roboflow datasets

In [ ]:
def copy_dataset(src: Path, dst: Path) -> Path:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    return dst

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
manifest = {"player": None, "logos": []}

print("Downloading player dataset...")
player_ds = (
    rf.workspace(PLAYER_WORKSPACE)
    .project(PLAYER_PROJECT)
    .version(PLAYER_VERSION)
    .download("yolov8")
)
player_src = Path(getattr(player_ds, "location", "soccer-iqjx5-2"))
player_dst = copy_dataset(player_src, ROBOFLOW_DIR / "player_soccer")
manifest["player"] = str(player_dst.resolve())
print("Player ->", player_dst)

for name, workspace, project, version in LOGO_DETECTION_SOURCES:
    print(f"Downloading {name}...")
    ds = rf.workspace(workspace).project(project).version(version).download("yolov8")
    src = Path(ds.location)
    dst = copy_dataset(src, ROBOFLOW_DIR / name)
    manifest["logos"].append({"name": name, "path": str(dst.resolve())})
    print(f"{name} ->", dst)

manifest_path = ROBOFLOW_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("Wrote", manifest_path)

## 3. Merge logo datasets → 1-class (`logo`)

In [ ]:
def iter_images(directory: Path):
    if not directory.exists():
        return
    for path in sorted(directory.iterdir()):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            yield path


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_as_class_agnostic(src_root: Path, dst_root: Path, prefix: str) -> dict:
    stats = {"images": 0, "labels": 0}
    for split in ("train", "valid", "test"):
        src_img = src_root / split / "images"
        if not src_img.exists():
            continue
        src_lbl = src_root / split / "labels"
        dst_img = ensure_dir(dst_root / split / "images")
        dst_lbl = ensure_dir(dst_root / split / "labels")
        for img_path in iter_images(src_img):
            new_stem = f"{prefix}__{img_path.stem}"
            shutil.copy2(img_path, dst_img / f"{new_stem}{img_path.suffix}")
            stats["images"] += 1
            label_path = src_lbl / f"{img_path.stem}.txt"
            out_lines = []
            if label_path.exists():
                for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
                    parts = line.split()
                    if len(parts) >= 5:
                        parts[0] = "0"
                        out_lines.append(" ".join(parts))
            (dst_lbl / f"{new_stem}.txt").write_text(
                "\n".join(out_lines) + ("\n" if out_lines else ""),
                encoding="utf-8",
            )
            stats["labels"] += len(out_lines)
    return stats


if MERGED_LOGO_DIR.exists():
    shutil.rmtree(MERGED_LOGO_DIR)
ensure_dir(MERGED_LOGO_DIR)

merge_stats = {}
for entry in manifest["logos"]:
    merge_stats[entry["name"]] = copy_as_class_agnostic(
        Path(entry["path"]), MERGED_LOGO_DIR, prefix=entry["name"]
    )

val_images = MERGED_LOGO_DIR / "valid" / "images"
if not val_images.exists() or not any(val_images.iterdir()):
    val_images = MERGED_LOGO_DIR / "train" / "images"

merged_yaml = {
    "train": str((MERGED_LOGO_DIR / "train" / "images").resolve()),
    "val": str(val_images.resolve()),
    "test": str((MERGED_LOGO_DIR / "test" / "images").resolve())
    if (MERGED_LOGO_DIR / "test" / "images").exists()
    else str(val_images.resolve()),
    "nc": 1,
    "names": ["logo"],
}
yaml_path = MERGED_LOGO_DIR / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(merged_yaml, f, allow_unicode=True, sort_keys=False)

print("Merged:", merge_stats)
print("data.yaml ->", yaml_path)

## 4. Train logo detector (5 epochs)

In [ ]:
logo_model = YOLO("yolov8n.pt")
logo_model.train(
    data=str(MERGED_LOGO_DIR / "data.yaml"),
    epochs=LOGO_EPOCHS,
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="logo_detector",
    exist_ok=True,
    verbose=True,
)

best = RUNS_DIR / "logo_detector" / "weights" / "best.pt"
assert best.exists(), best
LOGO_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, LOGO_WEIGHTS)
print("✅ Logo weights ->", LOGO_WEIGHTS)

## 5. Train player detector (25 epochs)

In [ ]:
def yolo_class_names(data_yaml: dict) -> list:
    names = data_yaml.get("names", [])
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names)


player_dir = Path(manifest["player"])
data_yaml_path = player_dir / "data.yaml"
with open(data_yaml_path, encoding="utf-8") as f:
    player_yaml = yaml.safe_load(f) or {}

player_names = yolo_class_names(player_yaml)
player_indices = [
    idx
    for idx, name in enumerate(player_names)
    if str(name).strip().lower() in PLAYER_NAME_HINTS
]
if not player_indices:
    print("No joueur/gardien/player match; training all classes.")
    player_indices = list(range(len(player_names)))

print("Player classes:", [(i, player_names[i]) for i in player_indices])

player_model = YOLO("yolov8n.pt")
player_model.train(
    data=str(data_yaml_path),
    epochs=PLAYER_EPOCHS,
    imgsz=IMGSZ,
    device=DEVICE,
    classes=player_indices,
    project=str(RUNS_DIR),
    name="player_detector",
    exist_ok=True,
    verbose=True,
)

best = RUNS_DIR / "player_detector" / "weights" / "best.pt"
assert best.exists(), best
PLAYER_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, PLAYER_WEIGHTS)
print("✅ Player weights ->", PLAYER_WEIGHTS)

## 6. Zip + download weights

After download, on your Mac:

```bash
cd ~/Desktop/csc490/local-data/groundingdino/data_collection/weights
mkdir -p yolo
unzip ~/Downloads/yolo_bakeoff_weights.zip -d yolo
# expect:
#   yolo/player_detector/best.pt
#   yolo/logo_detector/best.pt
```

Then locally:

```bash
cd ~/Desktop/csc490/csc490-group2
source .venv/bin/activate
cd groundingdino/data_collection

python yolo/yolo_labeler_player.py --directory athletico_real_2023/close_ups --limit 50
python yolo/yolo_labeler_logo.py --directory athletico_real_2023/close_ups --limit 50
```

In [ ]:
from google.colab import files

assert PLAYER_WEIGHTS.exists(), PLAYER_WEIGHTS
assert LOGO_WEIGHTS.exists(), LOGO_WEIGHTS

zip_path = ROOT / "yolo_bakeoff_weights"
archive = shutil.make_archive(str(zip_path), "zip", root_dir=WEIGHTS_OUT)
print("Created", archive)
print("Contents:")
for p in sorted(WEIGHTS_OUT.rglob("*.pt")):
    print(" ", p, f"({p.stat().st_size / 1e6:.1f} MB)")

if MOUNT_DRIVE:
    drive_dst = Path("/content/drive/MyDrive/yolo_bakeoff_weights.zip")
    shutil.copy2(archive, drive_dst)
    print("Also copied to", drive_dst)

files.download(archive)
print("✅ Download started — save the zip, then unzip into local-data/.../weights/yolo/")

## Optional: smoke-test one image in Colab

Upload a close-up JPG first, or skip this section.

In [ ]:
# Optional smoke test — set TEST_IMAGE to a path, or upload via the Files pane.
TEST_IMAGE = None  # e.g. "/content/frame_00000.jpg"

if TEST_IMAGE:
    from IPython.display import Image as IPyImage, display
    import cv2

    img = cv2.imread(TEST_IMAGE)
    player = YOLO(str(PLAYER_WEIGHTS))
    logo = YOLO(str(LOGO_WEIGHTS))
    pr = player(img, verbose=False)[0]
    lr = logo(img, verbose=False)[0]
    out = ROOT / "smoke_test.jpg"
    # draw player then logo
    plotted = pr.plot()
    cv2.imwrite(str(out), plotted)
    print("players:", 0 if pr.boxes is None else len(pr.boxes))
    print("logos:", 0 if lr.boxes is None else len(lr.boxes))
    display(IPyImage(filename=str(out)))
else:
    print("Skipped smoke test (set TEST_IMAGE to a JPG path).")